# Phase 6: Modeling V2 — XGBoost-First + Stacking

**Author:** Taline Zeidan  
**Course:** COE 546 — Machine Learning, Spring 2026  
**Input:** `data/train_features_v2.csv`, `data/test_features_v2.csv`  
**Output:** `outputs/submission_v2.csv`

---

## What Changed from V1

| Change | Reason |
|---|---|
| XGBoost is primary model (200 Optuna trials, 5-fold inner CV) | XGBoost alone (0.97031 LB) beat the V1 ensemble (0.96988 LB) |
| Stacking meta-learner instead of weighted average | Learns optimal combination rather than assuming CV AUC proportionality |
| V2 feature set (target encoding + log transforms + zero flags) | Richer features = better signal for all models |
| LGB + CatBoost: 100 Optuna trials each | Kept for ensemble diversity, not as primary models |

## 1. Imports & Setup

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
OUT_DIR    = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, 'train_features_v2.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test_features_v2.csv')
SUB_PATH   = os.path.join(OUT_DIR,  'submission_v2.csv')

# ── Constants ───────────────────────────────────────────────────────────────
SEED             = 42
N_FOLDS          = 5
SCALE_POS_WEIGHT = 33.4
N_TRIALS_XGB     = 200   # XGBoost is our primary — give it the most trials
N_TRIALS_LGB     = 100
N_TRIALS_CB      = 100
EARLY_STOPPING   = 50

print('All imports successful.')
print(f'LightGBM : {lgb.__version__}')
print(f'XGBoost  : {xgb.__version__}')
print(f'CatBoost : {cb.__version__}')
print(f'Optuna   : {optuna.__version__}')

All imports successful.
LightGBM : 4.3.0
XGBoost  : 2.0.3
CatBoost : 1.2.10
Optuna   : 4.8.0


## 2. Load Data

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

FEATURE_COLS = [c for c in train.columns if c not in ['id', 'order_placed']]

X      = train[FEATURE_COLS]
y      = train['order_placed']
X_test = test[FEATURE_COLS]

print(f'Train : {X.shape}  |  Positives: {y.sum():,} ({y.mean()*100:.2f}%)')
print(f'Test  : {X_test.shape}')
print(f'\nFeatures ({len(FEATURE_COLS)}):')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

Train : (297236, 52)  |  Positives: 8,644 (2.91%)
Test  : (99639, 52)

Features (52):
   1. timezone
   2. action_type
   3. promos_declined
   4. customer_type
   5. items_in_cart
   6. cart_value
   7. promo_type
   8. discount_value
   9. promos_shown
  10. screen_size
  11. promo_response
  12. session_duration_s
  13. inactivity_gap_s
  14. hour_of_day
  15. day_of_week
  16. is_weekend
  17. has_promo
  18. action_type_te
  19. customer_type_te
  20. promo_type_te
  21. promo_response_te
  22. timezone_te
  23. log_session_duration_s
  24. log_promos_shown
  25. log_cart_value
  26. log_items_in_cart
  27. log_screen_size
  28. log_discount_value
  29. cart_value_is_zero
  30. items_in_cart_is_zero
  31. promos_shown_is_zero
  32. session_duration_clipped
  33. log_session_duration_clipped
  34. is_default_duration
  35. engagement_ratio
  36. active_time_s
  37. promo_accepted
  38. is_promo_type_S
  39. discount_attractiveness
  40. cart_qualifies_for_promo
  41. is_returning_c

## 3. Cross-Validation Helper

In [3]:
def run_cv(model_fn, X, y, X_test, n_folds=N_FOLDS, seed=SEED, label='Model'):
    skf        = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs  = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        oof_fold, test_fold    = model_fn(X_tr, y_tr, X_val, y_val)
        oof_preds[val_idx]     = oof_fold
        test_preds            += test_fold / n_folds

        fold_auc = roc_auc_score(y_val, oof_fold)
        fold_aucs.append(fold_auc)
        print(f'  [{label}] Fold {fold}/{n_folds} — AUC: {fold_auc:.5f}')

    mean_auc = np.mean(fold_aucs)
    std_auc  = np.std(fold_aucs)
    oof_auc  = roc_auc_score(y, oof_preds)
    print(f'  [{label}] OOF AUC: {oof_auc:.5f} | Mean: {mean_auc:.5f} ± {std_auc:.5f}')
    return oof_preds, test_preds, mean_auc, std_auc

print('CV helper defined.')

CV helper defined.


## 4. Optuna Tuning — XGBoost (Primary Model)

XGBoost is our strongest model from V1. We give it **200 trials with 5-fold inner CV** for the most reliable hyperparameter search.

In [4]:
def xgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 2000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 100),
        'subsample'        : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma'            : trial.suggest_float('gamma', 0.0, 5.0),
        'scale_pos_weight' : SCALE_POS_WEIGHT,
        'eval_metric'      : 'auc',
        'use_label_encoder': False,
        'random_state'     : SEED,
        'n_jobs'           : -1,
        'verbosity'        : 0,
    }
    # 5-fold inner CV for more reliable signal
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set              = [(X_val, y_val)],
            early_stopping_rounds = EARLY_STOPPING,
            verbose               = False
        )
        aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(aucs)

print(f'Running Optuna on XGBoost ({N_TRIALS_XGB} trials × 5-fold)...')
print('This is the most important tuning step — be patient.')
study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params.update({
    'scale_pos_weight' : SCALE_POS_WEIGHT,
    'eval_metric'      : 'auc',
    'use_label_encoder': False,
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbosity'        : 0,
})
print(f'\n✅ Best XGBoost params: {best_xgb_params}')
print(f'   Best trial AUC     : {study_xgb.best_value:.5f}')

Running Optuna on XGBoost (200 trials × 5-fold)...
This is the most important tuning step — be patient.


Best trial: 190. Best value: 0.970226: 100%|██████████| 200/200 [10:16:05<00:00, 184.83s/it] 


✅ Best XGBoost params: {'n_estimators': 1359, 'learning_rate': 0.011736160425380787, 'max_depth': 11, 'min_child_weight': 50, 'subsample': 0.9353039427602737, 'colsample_bytree': 0.9781702713529885, 'colsample_bylevel': 0.9536159290816176, 'reg_alpha': 1.3890066998578428e-06, 'reg_lambda': 4.951691332870596e-06, 'gamma': 2.0693487206082963, 'scale_pos_weight': 33.4, 'eval_metric': 'auc', 'use_label_encoder': False, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
   Best trial AUC     : 0.97023


In [5]:
def xgb_tuned_fn(X_tr, y_tr, X_val, y_val):
    model = xgb.XGBClassifier(**best_xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set              = [(X_val, y_val)],
        early_stopping_rounds = EARLY_STOPPING,
        verbose               = False
    )
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]

print('Running tuned XGBoost (5-fold CV)...')
oof_xgb, test_xgb, auc_xgb, _ = run_cv(xgb_tuned_fn, X, y, X_test, label='XGB-V2')
print(f'\n✅ XGBoost V2 CV AUC : {auc_xgb:.5f}')
print(f'   XGBoost V1 CV AUC : 0.97045')
print(f'   V2 improvement    : {auc_xgb - 0.97045:+.5f}')

Running tuned XGBoost (5-fold CV)...
  [XGB-V2] Fold 1/5 — AUC: 0.96948
  [XGB-V2] Fold 2/5 — AUC: 0.97044
  [XGB-V2] Fold 3/5 — AUC: 0.96908
  [XGB-V2] Fold 4/5 — AUC: 0.97076
  [XGB-V2] Fold 5/5 — AUC: 0.97137
  [XGB-V2] OOF AUC: 0.96934 | Mean: 0.97023 ± 0.00084

✅ XGBoost V2 CV AUC : 0.97023
   XGBoost V1 CV AUC : 0.97045
   V2 improvement    : -0.00022


## 5. Optuna Tuning — LightGBM

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 2000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 400),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample'        : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain'   : trial.suggest_float('min_split_gain', 0.0, 1.0),
        'scale_pos_weight' : SCALE_POS_WEIGHT,
        'n_jobs'           : -1,
        'random_state'     : SEED,
        'verbose'          : -1,
    }
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set  = [(X_val, y_val)],
            callbacks = [lgb.early_stopping(EARLY_STOPPING, verbose=False),
                         lgb.log_evaluation(-1)]
        )
        aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(aucs)

print(f'Running Optuna on LightGBM ({N_TRIALS_LGB} trials × 5-fold)...')
study_lgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS_LGB, show_progress_bar=True)

best_lgb_params = study_lgb.best_params
best_lgb_params.update({
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'n_jobs': -1, 'random_state': SEED, 'verbose': -1
})
print(f'\n✅ Best LGB params  : {best_lgb_params}')
print(f'   Best trial AUC  : {study_lgb.best_value:.5f}')

Running Optuna on LightGBM (100 trials × 5-fold)...


Best trial: 85. Best value: 0.967538: 100%|██████████| 100/100 [28:15<00:00, 16.95s/it]


✅ Best LGB params  : {'n_estimators': 1216, 'learning_rate': 0.005494198003025223, 'num_leaves': 249, 'min_child_samples': 153, 'subsample': 0.5070542706621705, 'colsample_bytree': 0.4043274726111492, 'reg_alpha': 1.2517896604774755e-08, 'reg_lambda': 0.0019764952218428193, 'min_split_gain': 0.4395181384311467, 'scale_pos_weight': 33.4, 'n_jobs': -1, 'random_state': 42, 'verbose': -1}
   Best trial AUC  : 0.96754


In [ ]:
def lgb_tuned_fn(X_tr, y_tr, X_val, y_val):
    model = lgb.LGBMClassifier(**best_lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set  = [(X_val, y_val)],
        callbacks = [lgb.early_stopping(EARLY_STOPPING, verbose=False),
                     lgb.log_evaluation(-1)]
    )
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]

print('Running tuned LightGBM (5-fold CV)...')
oof_lgb, test_lgb, auc_lgb, _ = run_cv(lgb_tuned_fn, X, y, X_test, label='LGB-V2')
print(f'\n✅ LightGBM V2 CV AUC: {auc_lgb:.5f}')

Running tuned LightGBM (5-fold CV)...
  [LGB-V2] Fold 1/5 — AUC: 0.96634
  [LGB-V2] Fold 2/5 — AUC: 0.96741
  [LGB-V2] Fold 3/5 — AUC: 0.96702
  [LGB-V2] Fold 4/5 — AUC: 0.96794
  [LGB-V2] Fold 5/5 — AUC: 0.96898
  [LGB-V2] OOF AUC: 0.96710 | Mean: 0.96754 ± 0.00089

✅ LightGBM V2 CV AUC: 0.96754


## 6. Optuna Tuning — CatBoost

In [ ]:
def cb_objective(trial):
    params = {
        'iterations'         : trial.suggest_int('iterations', 300, 2000),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'depth'              : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength'    : trial.suggest_float('random_strength', 0.0, 2.0),
        'border_count'       : trial.suggest_int('border_count', 32, 255),
        'class_weights'      : [1, SCALE_POS_WEIGHT],
        'eval_metric'        : 'AUC',
        'random_seed'        : SEED,
        'verbose'            : False,
    }
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set              = (X_val, y_val),
            early_stopping_rounds = EARLY_STOPPING,
            verbose               = False
        )
        aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    return np.mean(aucs)

print(f'Running Optuna on CatBoost ({N_TRIALS_CB} trials × 5-fold)...')
study_cb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_cb.optimize(cb_objective, n_trials=N_TRIALS_CB, show_progress_bar=True)

best_cb_params = study_cb.best_params
best_cb_params.update({
    'class_weights': [1, SCALE_POS_WEIGHT],
    'eval_metric'  : 'AUC',
    'random_seed'  : SEED,
    'verbose'      : False,
})
print(f'\n✅ Best CatBoost params: {best_cb_params}')
print(f'   Best trial AUC     : {study_cb.best_value:.5f}')

Running Optuna on CatBoost (100 trials × 5-fold)...


Best trial: 1. Best value: 0.967682:   2%|▏         | 2/100 [01:45<1:32:16, 56.49s/it]

In [ ]:
def cb_tuned_fn(X_tr, y_tr, X_val, y_val):
    model = cb.CatBoostClassifier(**best_cb_params)
    model.fit(
        X_tr, y_tr,
        eval_set              = (X_val, y_val),
        early_stopping_rounds = EARLY_STOPPING,
        verbose               = False
    )
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]

print('Running tuned CatBoost (5-fold CV)...')
oof_cb, test_cb, auc_cb, _ = run_cv(cb_tuned_fn, X, y, X_test, label='CB-V2')
print(f'\n✅ CatBoost V2 CV AUC: {auc_cb:.5f}')

## 7. Stacking Meta-Learner

Instead of a weighted average, we train a **Logistic Regression** on the out-of-fold predictions of our three base models. This is called **stacking** (or stacked generalization).

**Why stacking beats weighted average:**
- Weighted average assumes the optimal combination is proportional to CV AUC — a simplification
- Logistic regression *learns* the optimal combination from data
- It can even learn that one model should be down-weighted in certain probability ranges

**Why Logistic Regression as meta-learner:**
- Simple, fast, interpretable
- Unlikely to overfit on 3 features (the 3 OOF prediction columns)
- Works well when base model predictions are already well-calibrated probabilities

**Leakage prevention:**
- Meta-learner is trained only on OOF predictions (never saw its own training labels during base model training)
- Test predictions are the averaged test predictions from each fold of the base models

In [ ]:
# ── Build meta-features ───────────────────────────────────────────────────
oof_meta  = np.column_stack([oof_xgb, oof_lgb, oof_cb])   # shape: (n_train, 3)
test_meta = np.column_stack([test_xgb, test_lgb, test_cb]) # shape: (n_test, 3)

print('Meta-feature shapes:')
print(f'  OOF  (train): {oof_meta.shape}')
print(f'  Test        : {test_meta.shape}')

# ── Scale (Logistic Regression is sensitive to scale) ─────────────────────
scaler    = StandardScaler()
oof_meta_scaled  = scaler.fit_transform(oof_meta)
test_meta_scaled = scaler.transform(test_meta)

# ── Train meta-learner ────────────────────────────────────────────────────
meta_model = LogisticRegression(
    C                = 1.0,
    max_iter         = 1000,
    random_state     = SEED,
    class_weight     = 'balanced'
)
meta_model.fit(oof_meta_scaled, y)

# ── OOF AUC of stacked model ──────────────────────────────────────────────
stack_oof_preds  = meta_model.predict_proba(oof_meta_scaled)[:, 1]
stack_test_preds = meta_model.predict_proba(test_meta_scaled)[:, 1]
stack_auc        = roc_auc_score(y, stack_oof_preds)

# ── Compare with simple weighted average ──────────────────────────────────
aucs    = np.array([auc_xgb, auc_lgb, auc_cb])
weights = aucs / aucs.sum()
wavg_oof_preds  = weights[0]*oof_xgb  + weights[1]*oof_lgb  + weights[2]*oof_cb
wavg_test_preds = weights[0]*test_xgb + weights[1]*test_lgb + weights[2]*test_cb
wavg_auc        = roc_auc_score(y, wavg_oof_preds)

print(f'\nBase model CV AUCs:')
print(f'  XGBoost  : {auc_xgb:.5f} (weight in wavg: {weights[0]:.4f})')
print(f'  LightGBM : {auc_lgb:.5f} (weight in wavg: {weights[1]:.4f})')
print(f'  CatBoost : {auc_cb:.5f} (weight in wavg: {weights[2]:.4f})')
print(f'\nEnsemble comparison:')
print(f'  Weighted average OOF AUC : {wavg_auc:.5f}')
print(f'  Stacking OOF AUC         : {stack_auc:.5f}')
print(f'  Stacking gain            : {stack_auc - wavg_auc:+.5f}')

print(f'\nMeta-learner coefficients (XGB, LGB, CB):')
print(f'  {meta_model.coef_[0].round(4)}')

## 8. Choose Best Submission

We compare stacking vs weighted average vs XGBoost alone and submit the best.

In [ ]:
results = {
    'XGBoost only'    : (roc_auc_score(y, oof_xgb),         test_xgb),
    'Weighted average': (wavg_auc,                           wavg_test_preds),
    'Stacking'        : (stack_auc,                          stack_test_preds),
}

print('OOF AUC comparison:')
best_name, best_auc, best_preds = None, 0, None
for name, (auc, preds) in results.items():
    marker = ' ← BEST' if auc == max(r[0] for r in results.values()) else ''
    print(f'  {name:20s}: {auc:.5f}{marker}')
    if auc > best_auc:
        best_auc, best_name, best_preds = auc, name, preds

print(f'\nSubmitting: {best_name} (OOF AUC: {best_auc:.5f})')

## 9. Feature Importance (XGBoost V2)

In [ ]:
final_xgb = xgb.XGBClassifier(**best_xgb_params)
final_xgb.fit(X, y, verbose=False)

importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': final_xgb.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, len(FEATURE_COLS) * 0.28))
sns.barplot(data=importance_df, x='importance', y='feature', color='steelblue', ax=ax)
ax.set_title('XGBoost V2 Feature Importance', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'feature_importance_v2.png'), dpi=150)
plt.show()

print('Top 15 features:')
print(importance_df.head(15).to_string(index=False))

## 10. Save Both Submissions

In [ ]:
# ── Primary submission (best ensemble) ────────────────────────────────────
submission_v2 = pd.DataFrame({
    'id'          : test['id'],
    'order_placed': best_preds
})
submission_v2.to_csv(SUB_PATH, index=False)
print(f'✅ Primary submission saved: {SUB_PATH}')
print(f'   Method: {best_name}')
print(f'   OOF AUC: {best_auc:.5f}')

# ── Also save XGBoost-only as fallback ────────────────────────────────────
xgb_only_path = os.path.join(OUT_DIR, 'submission_v2_xgb_only.csv')
pd.DataFrame({'id': test['id'], 'order_placed': test_xgb}).to_csv(xgb_only_path, index=False)
print(f'\n✅ XGBoost-only fallback saved: {xgb_only_path}')

# ── Also save stacking ────────────────────────────────────────────────────
stack_path = os.path.join(OUT_DIR, 'submission_v2_stacking.csv')
pd.DataFrame({'id': test['id'], 'order_placed': stack_test_preds}).to_csv(stack_path, index=False)
print(f'✅ Stacking submission saved  : {stack_path}')

print(f'\nSample predictions:')
print(submission_v2.head(10).to_string(index=False))

## 11. Results Summary

In [ ]:
print('=' * 60)
print('V2 MODELING RESULTS SUMMARY')
print('=' * 60)
print(f'  Feature set       : V2 ({len(FEATURE_COLS)} features)')
print(f'  XGBoost  CV AUC   : {auc_xgb:.5f}  (200 Optuna trials, 5-fold)')
print(f'  LightGBM CV AUC   : {auc_lgb:.5f}  (100 Optuna trials, 5-fold)')
print(f'  CatBoost CV AUC   : {auc_cb:.5f}  (100 Optuna trials, 5-fold)')
print('-' * 60)
print(f'  Weighted avg AUC  : {wavg_auc:.5f}')
print(f'  Stacking AUC      : {stack_auc:.5f}')
print('-' * 60)
print(f'  SUBMITTED         : {best_name} → {best_auc:.5f}')
print('=' * 60)
print(f'\n  V1 public LB      : 0.97031  (XGBoost only)')
print(f'  V2 target         : > 0.97100')
print(f'  Leaderboard top   : 0.97188')